**Why Deep Research Agents Matter:**

- **Market Intelligence**: Automatically gather competitive insights, industry trends, and market dynamics
- **Due Diligence**: Research potential partners, investments, or acquisition targets with comprehensive data collection
- **Content Creation**: Generate well-researched reports, whitepapers, and strategic documents
- **Decision Support**: Provide data-driven insights for executive decision-making
- **Customer Research**: Understand target audiences, pain points, and market opportunities

**Personal Applications:**
- Academic research and paper writing
- Investment research and analysis  
- Career planning and industry exploration
- Product comparisons and purchasing decisions
- Learning new domains quickly and thoroughly

The versatility of Deep Research agents makes them one of the most valuable AI implementations across industries and personal workflows.

In [1]:
from agents import Agent, trace, Runner, gen_trace_id, function_tool, OpenAIChatCompletionsModel, WebSearchTool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown
from openai import AsyncOpenAI

In [2]:
# === LLM SETUP (edit only this cell / .env to change provider/model) ===
# Defaults to Ollama.
load_dotenv(override=True)

LOCAL_LLM_API_KEY = os.getenv("LOCAL_LLM_API_KEY", "ollama")
LOCAL_LLM_BASE_URL = os.getenv("LOCAL_LLM_BASE_URL", "http://localhost:11434/v1")
LOCAL_LLM_MODEL = os.getenv("LOCAL_LLM_MODEL", "llama3.2:latest")
print(f"Using model: {LOCAL_LLM_MODEL} @ {LOCAL_LLM_BASE_URL}")

## OpenAI Hosted Tools

The OpenAI Agents SDK includes several powerful hosted tools to extend agent capabilities:

### Available Hosted Tools

****  
Enables agents to search the web for current information and real-time data.

****  
Allows agents to retrieve information from your OpenAI Vector Stores for knowledge base queries.

****  
Enables computer automation tasks like taking screenshots, clicking elements, and interacting with applications.

---

### ⚠️ Important Cost Consideration - WebSearchTool

The  has significant usage costs:
- **/bin/zsh.025 per search call** (2.5 cents)
- This can quickly add up to **-** over the next 2 labs
- See full pricing: [OpenAI Web Search Pricing](https://platform.openai.com/docs/pricing#web-search)

**Recommendation:** If cost is a concern, feel free to **skip running** the WebSearchTool examples. We will explore low-cost search alternatives with other platforms in upcoming labs.

**Alternative Approach:** For learning purposes, you can simulate the web search results or use free/lower-cost search APIs during development and testing.

In [ ]:
local_openai_client = AsyncOpenAI(base_url=LOCAL_LLM_BASE_URL, api_key=LOCAL_LLM_API_KEY)
local_openai_model = OpenAIChatCompletionsModel(model=LOCAL_LLM_MODEL, openai_client=local_openai_client)

In [4]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model=local_openai_model,
    model_settings=ModelSettings(tool_choice="required"),
)

In [5]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

# Create simulated web search tool
@function_tool
async def simulated_web_search(query: str) -> str:
    """Simulate web search results for testing purposes"""
    mock_results = {
        "Latest AI Agent frameworks in 2025": """
        Based on current trends, the latest AI Agent frameworks in 2025 include:

        1. **OpenAI Agents SDK** - Comprehensive framework with hosted tools and function calling
        2. **LangChain** - Modular approach with extensive integrations and RAG capabilities
        3. **AutoGen** - Microsofts multi-agent conversation framework with group chat
        4. **CrewAI** - Task-based agent orchestration with role-based workflows
        5. **LangGraph** - Stateful, multi-actor applications with complex workflows
        6. **Semantic Kernel** - Microsofts AI orchestration framework
        7. **Flowise** - Low-code platform for building AI agents

        Key trends: Increased focus on tool integration, better memory management,
        improved agent coordination, multimodal capabilities, and cost optimization.
        """,
        "AI development trends": """
        Current AI development trends focus on:
        - Multimodal agents (text, image, audio, video)
        - Better reasoning and planning capabilities
        - Cost optimization and efficiency
        - Improved memory and context management
        - Enhanced tool integration and API connectivity
        - Better human-AI collaboration interfaces
        """,
        "AI agent market": """
        The AI agent market is experiencing rapid growth with:
        - Enterprise adoption for automation and decision support
        - Integration with existing business workflows
        - Focus on domain-specific applications
        - Increased investment in agent infrastructure
        - Growing ecosystem of specialized tools and platforms
        """
    }

    # Return mock results or a generic response
    # ✅ Fixed properly closed f-string
    return mock_results.get(
        query,
        f"Simulated search results for: {query}\n\n"
        "This is a mock response for testing purposes. In a real implementation, "
        "this would contain actual web search results."
    )

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[simulated_web_search],  # Use simulated search instead of WebSearchTool
    model=local_openai_model,
    model_settings=ModelSettings(tool_choice="required"),
)

In [7]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

AI Agent frameworks 2025:

- **LangChain v0.3** – new modular agent core, built‑in planner, memory adapters, integration with OpenAI, Anthropic, Cohere, LlamaIndex. Supports multi‑modal input (text+image) and chain of thought prompting.  
- **Haystack Agents** – open‑source framework for search‑retrieval augmented agents; uses vector DBs (Weaviate, Pinecone), retrieval pipelines, auto‑scoring. Adds conversational state persistence.  
- **OpenAI Agent SDK** – official wrapper around GPT‑4o and GPT‑5 models, offers built‑in task orchestration, tool calling, and safety hooks. Works with LangSmith for observability.  
- **Agentic LLMs by Cohere** – Cohere’s “Cohere Agent” uses prompt‑based planning + vector memory; supports fine‑tuning on custom workflows.  
- **DeepMind Gemini Agents** – Gemini 1.5 agents expose tool calling API, multi‑modal reasoning, and in‑house policy enforcement for safety.  
- **Microsoft Azure AI Agents** – built into Azure OpenAI Service, includes workflow designer, stateful memory, and compliance controls for enterprise use.  

Common trends: plug‑and‑play memory modules, automatic plan generation, tool calling with sandboxed APIs, observability dashboards, and emphasis on safety/policy enforcement. These frameworks lower entry barrier for building complex, multi‑step agents in production.

### As always, take a look at the trace

https://platform.openai.com/traces

### Next: Implementing Structured Outputs with Field Descriptions

We'll now leverage **Structured Outputs** to ensure our agent returns data in a consistent, predictable format. By including detailed field descriptions, we can guide the model to produce exactly the structure we need for downstream processing.

This approach provides:
- **Guaranteed schema compliance** - no more parsing errors
- **Clear field definitions** - the model understands exactly what each field should contain
- **Type safety** - structured data that integrates seamlessly with your application logic

In [14]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 3

# INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
# to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

INSTRUCTIONS = f"""
You are a helpful research assistant. Given a query, come up with {HOW_MANY_SEARCHES} web searches
to perform to best answer the query. Output exactly {HOW_MANY_SEARCHES} search items as a JSON array, formatted as:

[
  {{
    "reason": "Your reasoning for why this search is important to the query.",
    "query": "The search term to use for the web search."
  }},
  {{
    "reason": "...",
    "query": "..."
  }},
  {{
    "reason": "...",
    "query": "..."
  }}
]

Do not include any commentary or text outside the JSON array.
"""

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model=local_openai_model,
    output_type=WebSearchPlan,
)

In [15]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='Identify the most recent AI agent framework releases and their key features for 2025.', query='latest AI agent framework 2025 release notes'), WebSearchItem(reason='Compare current AI agent frameworks to existing literature on performance metrics and evaluation', query='AI agent framework comparison 2024-2023?')]


In [16]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("contact@alakinfotech.com")  # Change to your verified sender
    to_email = To("reddy.revanth@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [17]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given subject and HTML body', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10fd55f80>, strict_json_schema=True, is_enabled=True)

In [18]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model=local_openai_model,
)

In [19]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model=local_openai_model,
    output_type=ReportData,
)

### Multi-Agent Search Architecture: Planning and Execution

The next 3 functions demonstrate a **collaborative multi-agent approach** where specialized agents work together:

- **`planner_agent`** - Analyzes the research request and creates a strategic search plan
- **`search_agent`** - Executes the planned searches and gathers relevant information  
- **Orchestration layer** - Coordinates between planning and execution phases

This division of labor ensures more targeted, efficient research by separating strategy (what to search) from execution (how to search and retrieve data).

In [20]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### Report Generation and Delivery Pipeline

The next 2 functions complete the research workflow by:

- **Report Writing Function** - Synthesizes gathered research into a comprehensive, well-structured document
- **Email Delivery Function** - Automatically sends the completed report to stakeholders via email

This creates an end-to-end research automation pipeline: from planning and data collection to final report delivery.

In [21]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### Let’s Execute the Workflow! 🚀

In [24]:
query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)
    print("Hooray!")

Starting research...
Planning searches...
Will perform 3 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!
